# Flight Delay Prediction: ML Modeling (Final, Production-Oriented)

This notebook trains and evaluates machine learning models using the **leakage-safe modeling datasets**
produced by:

- `data_cleaning_feature_engineering.ipynb`

## Inputs
From `outputs/`:
- `flight_delay_train_features.parquet`
- `flight_delay_test_features.parquet`

## Target
- `IS_DELAYED` (arrival delay >= 15 minutes)
- `IS_DELAYED_15` is treated as an alias where present

## Models
1. Logistic Regression (baseline, interpretable)
2. XGBoost 
3. CatBoost 

## Evaluation
Primary metric: **PR-AUC** (appropriate for imbalanced "delay" class)  
Secondary metrics: ROC-AUC, Brier score

In [1]:
import os
import joblib
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import confusion_matrix
from sklearn.calibration import CalibratedClassifierCV

from xgboost import XGBClassifier


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier


from dotenv import load_dotenv

load_dotenv()

True

In [2]:
ART_DIR = os.getenv("ART_DIR")
META_PATH = os.getenv("META_PATH")
JOBLIB_PATH = os.getenv("JOBLIB_PATH")
CB_PATH = os.getenv("CB_PATH")
LOOKUP_PATH = os.getenv("LOOKUP_PATH")
FEATURE_STORE_PATH = os.getenv("FEATURE_STORE_PATH")

## 1) Load leakage-safe train/test feature tables


In [3]:
TRAIN_PATH = os.getenv("TRAIN_PATH")
TEST_PATH  = os.getenv("TEST_PATH")

train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
train_df.head()

Train shape: (118388, 70) Test shape: (29597, 70)


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,UNIQUE_CARRIER,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,...,ORIGIN_RISK,DEST_RISK,UNIQUE_CARRIER_RISK,ROUTE_RISK,CARRIER_ROUTE_RISK,TRAIN_ORIGIN_COUNT,TRAIN_DEST_COUNT,TRAIN_ROUTE_COUNT,TRAIN_UNIQUE_CARRIER_COUNT,TRAIN_CARRIER_ROUTE_COUNT
0,2017,2,5,1,1,2017-05-01,EV,ATL,"Atlanta, GA",GA,...,0.164977,0.198791,0.191210,0.198791,0.225986,58308,326,326,9148,148
1,2017,2,5,1,1,2017-05-01,DL,MLB,"Melbourne, FL",FL,...,0.106172,0.143650,0.130001,0.106172,0.106385,249,58354,249,77878,239
2,2017,2,5,1,1,2017-05-01,DL,TPA,"Tampa, FL",FL,...,0.174379,0.143650,0.130001,0.174379,0.126902,1084,58354,1084,77878,736
3,2017,2,5,1,1,2017-05-01,DL,GNV,"Gainesville, FL",FL,...,0.172097,0.143650,0.130001,0.172097,0.124940,210,58354,210,77878,28
4,2017,2,5,1,1,2017-05-01,DL,ATL,"Atlanta, GA",GA,...,0.164977,0.100682,0.130001,0.098811,0.116122,58308,603,585,77878,387


## 2) Define target, remove leakage columns, and select features

The modeling datasets were created to be leakage-safe, but still apply a final safeguard:
- Exclude `ARR_DELAY` if present (direct label leakage)
- Exclude raw `FL_DATE` from features (keep only for reporting)

Feature selection rationale:
- Using high-signal engineered features highlighted by EDA:
  - route/carrier/airport behavior via **risk encodings**
  - time-of-day and seasonal effects
  - distance/airtime characteristics
  - hub and traffic context as secondary signals

This avoids training on many redundant flags while keeping the strongest predictors.

In [4]:
# Target column
if "IS_DELAYED" in train_df.columns:
    TARGET = "IS_DELAYED"
elif "IS_DELAYED_15" in train_df.columns:
    TARGET = "IS_DELAYED_15"
else:
    raise KeyError("Target not found. Expected IS_DELAYED or IS_DELAYED_15.")

# Parse FL_DATE for reporting only
for df_ in (train_df, test_df):
    if "FL_DATE" in df_.columns:
        df_["FL_DATE"] = pd.to_datetime(df_["FL_DATE"], errors="coerce")

y_train = train_df[TARGET].astype(int).values
y_test  = test_df[TARGET].astype(int).values

print("Delay rate (train):", round(y_train.mean(), 4), "Delay rate (test):", round(y_test.mean(), 4))
if "FL_DATE" in train_df.columns:
    print("Train date range:", train_df["FL_DATE"].min(), "->", train_df["FL_DATE"].max())
if "FL_DATE" in test_df.columns:
    print("Test  date range:", test_df["FL_DATE"].min(), "->", test_df["FL_DATE"].max())

# Columns to always exclude from features (leakage safeguards)
ALWAYS_EXCLUDE = {TARGET, "IS_DELAYED_15", "ARR_DELAY", "FL_DATE"}
ALWAYS_EXCLUDE = {c for c in ALWAYS_EXCLUDE if c in train_df.columns}

# Candidate feature groups (use what exists)
CORE_TEMPORAL = [
    "MONTH", "DAY_OF_WEEK", "DAY_OF_MONTH", "QUARTER",
    "IS_WEEKEND", "SEASON", "IS_SUMMER", "IS_WINTER", "IS_HOLIDAY_SEASON",
    "DEP_HOUR_BIN"
]

FLIGHT_CHARACTERISTICS = [
    "DISTANCE", "AIR_TIME", "DISTANCE_GROUP", "DISTANCE_CAT", "DISTANCE_NORMALIZED",
    "IS_SHORT_HAUL", "IS_MEDIUM_HAUL", "IS_LONG_HAUL"
]

# Risk encodings created in the FE notebook (train-only, leakage-safe)
RISK_FEATURES = [c for c in train_df.columns if c.endswith("_RISK")]

# Train-only count features
COUNT_FEATURES = [c for c in train_df.columns if c.endswith("_COUNT")]

# Context / operational proxies
CONTEXT = [
    "IS_HUB_ORIGIN", "IS_HUB_DEST", "IS_HUB_TO_HUB",
    "IS_BUSY_ORIGIN", "IS_BUSY_DEST",
    "IS_POPULAR_ROUTE",
    "ROUTE_POPULARITY", "ORIGIN_TRAFFIC", "DEST_TRAFFIC", "CARRIER_VOLUME",
]

# Include raw categoricals for CatBoost; for LR/XGB they will be one-hot encoded.
RAW_CATEGORICALS = ["UNIQUE_CARRIER", "ORIGIN", "DEST", "ROUTE", "CARRIER_ROUTE",
                    "ORIGIN_STATE_ABR", "DEST_STATE_ABR"]

# Build selected feature list, keeping only those present
selected = []
for group in [CORE_TEMPORAL, FLIGHT_CHARACTERISTICS, CONTEXT, RAW_CATEGORICALS, RISK_FEATURES, COUNT_FEATURES]:
    for c in group:
        if c in train_df.columns and c not in ALWAYS_EXCLUDE:
            selected.append(c)

# De-duplicate while preserving order
selected = list(dict.fromkeys(selected))

# Final X matrices
X_train = train_df[selected].copy()
X_test  = test_df[selected].copy()

print("Selected features:", len(selected))
print("Risk features:", len(RISK_FEATURES), "Count features:", len(COUNT_FEATURES))
selected[:20]


Delay rate (train): 0.1549 Delay rate (test): 0.1291
Train date range: 2017-05-01 00:00:00 -> 2018-02-22 00:00:00
Test  date range: 2018-02-23 00:00:00 -> 2018-04-30 00:00:00


Selected features: 45
Risk features: 5 Count features: 5


['MONTH',
 'DAY_OF_WEEK',
 'DAY_OF_MONTH',
 'QUARTER',
 'IS_WEEKEND',
 'SEASON',
 'IS_SUMMER',
 'IS_WINTER',
 'IS_HOLIDAY_SEASON',
 'DEP_HOUR_BIN',
 'DISTANCE',
 'AIR_TIME',
 'DISTANCE_GROUP',
 'DISTANCE_CAT',
 'DISTANCE_NORMALIZED',
 'IS_SHORT_HAUL',
 'IS_MEDIUM_HAUL',
 'IS_LONG_HAUL',
 'IS_HUB_ORIGIN',
 'IS_HUB_DEST']

## 3) Preprocessing for Logistic Regression and XGBoost

- Numeric: median imputation + scaling (scaling is important for logistic regression)
- Categorical: most frequent imputation + one-hot encoding (sparse output)

CatBoost will bypass this and use the raw DataFrame with native categorical support.


In [5]:
# Identify categorical vs numeric by dtype (object/category treated as categorical)
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("category")]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

print("Numeric features:", len(num_cols), "Categorical features:", len(cat_cols))
cat_cols


Numeric features: 35 Categorical features: 10


['SEASON',
 'DEP_HOUR_BIN',
 'DISTANCE_CAT',
 'UNIQUE_CARRIER',
 'ORIGIN',
 'DEST',
 'ROUTE',
 'CARRIER_ROUTE',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR']

## 4) Train models and compute metrics


In [6]:
def compute_metrics(y_true, p):
    return {
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc": average_precision_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
    }

results = []
preds = {}  # store probabilities per model for later thresholding/calibration

# Logistic Regression baseline
lr = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=7000,
        solver="saga",
        n_jobs=-1,
        random_state=42
    ))
])

lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_test)[:, 1]
preds["LogisticRegression"] = p_lr
m_lr = compute_metrics(y_test, p_lr)
results.append({"model": "LogisticRegression", **m_lr})
m_lr


{'roc_auc': 0.6716150525429482,
 'pr_auc': 0.27850518622502357,
 'brier': 0.10945741669829424}

In [7]:
xgb_available = True

xgb = None
if xgb_available:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    xgb = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_weight=1.0,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1
        ))
    ])

    xgb.fit(X_train, y_train)
    p_xgb = xgb.predict_proba(X_test)[:, 1]
    preds["XGBoost"] = p_xgb
    m_xgb = compute_metrics(y_test, p_xgb)
    results.append({"model": "XGBoost", **m_xgb})
m_xgb


{'roc_auc': 0.6790848239197573,
 'pr_auc': 0.29026075525301165,
 'brier': 0.17971496535934176}

In [8]:
# CatBoost + Platt (sigmoid) probability calibration
# ---------------------------------------------------
# Why:
# - CatBoost often ranks well but can be overconfident in its raw probabilities.
# - Platt scaling makes probabilities more realistic for UI display (LOW/MODERATE/HIGH).
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
import numpy as np

cb_available = True
cb = None
platt = None  # <- saved and exported later if CatBoost is the best model

if cb_available:
    # Keep numeric columns numeric; only coerce categoricals to strings
    X_train_cb = X_train.copy()
    X_test_cb  = X_test.copy()

    # Use the same categorical columns we detected earlier (cat_cols)
    # Fill missing categoricals with a stable token
    for c in cat_cols:
        X_train_cb[c] = X_train_cb[c].astype(str).fillna("NA")
        X_test_cb[c]  = X_test_cb[c].astype(str).fillna("NA")

    # CatBoost needs categorical feature indices
    cat_feature_indices = [X_train_cb.columns.get_loc(c) for c in cat_cols]

    # Split TRAIN into fit/calibration sets (leakage-safe)
    X_fit_cb, X_cal_cb, y_fit, y_cal = train_test_split(
        X_train_cb, y_train,
        test_size=0.20,
        random_state=42,
        stratify=y_train
    )

    # Train CatBoost on X_fit and validate on X_cal
    cb = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=200
    )

    cb.fit(
        X_fit_cb, y_fit,
        cat_features=cat_feature_indices,
        eval_set=(X_cal_cb, y_cal),
        use_best_model=True
    )

    # ---- Platt scaling (sigmoid) on CAL set ----
    p_cal_raw = cb.predict_proba(X_cal_cb)[:, 1]

    eps = 1e-6
    log_odds_cal = np.log((p_cal_raw + eps) / (1 - p_cal_raw + eps)).reshape(-1, 1)

    platt = LogisticRegression(solver="lbfgs")
    platt.fit(log_odds_cal, y_cal)

    # ---- Evaluate on TEST (untouched) ----
    p_test_raw = cb.predict_proba(X_test_cb)[:, 1]
    log_odds_test = np.log((p_test_raw + eps) / (1 - p_test_raw + eps)).reshape(-1, 1)
    p_test_cal = platt.predict_proba(log_odds_test)[:, 1]

    print("CatBoost AUC raw:", roc_auc_score(y_test, p_test_raw))
    print("CatBoost AUC calibrated:", roc_auc_score(y_test, p_test_cal))

    print("CatBoost Brier raw:", brier_score_loss(y_test, p_test_raw))
    print("CatBoost Brier calibrated:", brier_score_loss(y_test, p_test_cal))

    # Store CALIBRATED probabilities for model selection + thresholding
    preds["CatBoost"] = p_test_cal
    m_cb = compute_metrics(y_test, p_test_cal)
    results.append({"model": "CatBoost", **m_cb})
    m_cb


0:	test: 0.6540783	best: 0.6540783 (0)	total: 330ms	remaining: 11m


200:	test: 0.7804002	best: 0.7804002 (200)	total: 46.6s	remaining: 6m 57s


400:	test: 0.7973608	best: 0.7973608 (400)	total: 1m 38s	remaining: 6m 34s


600:	test: 0.8014619	best: 0.8014923 (599)	total: 2m 27s	remaining: 5m 44s


800:	test: 0.8036022	best: 0.8036022 (800)	total: 3m 19s	remaining: 4m 58s


...

1400:	test: 0.8061665	best: 0.8061665 (1400)	total: 5m 49s	remaining: 2m 29s


1600:	test: 0.8065037	best: 0.8065233 (1599)	total: 6m 38s	remaining: 1m 39s


1800:	test: 0.8069062	best: 0.8069093 (1769)	total: 7m 33s	remaining: 50.1s


1999:	test: 0.8066491	best: 0.8070135 (1929)	total: 8m 35s	remaining: 0us

bestTest = 0.8070134507
bestIteration = 1929

Shrink model to first 1930 iterations.


CatBoost AUC raw: 0.6731748895848371
CatBoost AUC calibrated: 0.6731748895848371
CatBoost Brier raw: 0.11379467852999195
CatBoost Brier calibrated: 0.11550393858627388


In [9]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

y_pred = cb.predict(X_test_cb)
y_pred_proba = cb.predict_proba(X_test_cb)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8573504071358584
AUC: 0.6731748895848371


              precision    recall  f1-score   support

           0       0.89      0.96      0.92     25776
           1       0.39      0.19      0.26      3821

    accuracy                           0.86     29597
   macro avg       0.64      0.57      0.59     29597
weighted avg       0.82      0.86      0.84     29597

Confusion Matrix:
 [[24644  1132]
 [ 3090   731]]


In [10]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_pred_proba)
print("PR-AUC:", pr_auc)

PR-AUC: 0.2965938255350137


In [11]:
import numpy as np
from sklearn.metrics import classification_report

for t in [0.2, 0.25, 0.3, 0.35]:
    preds_t = (y_pred_proba >= t).astype(int)
    print(f"\nThreshold = {t}")
    print(classification_report(y_test, preds_t))


Threshold = 0.2


              precision    recall  f1-score   support

           0       0.91      0.73      0.81     25776
           1       0.22      0.51      0.31      3821

    accuracy                           0.71     29597
   macro avg       0.57      0.62      0.56     29597
weighted avg       0.82      0.71      0.75     29597




Threshold = 0.25


              precision    recall  f1-score   support

           0       0.91      0.81      0.86     25776
           1       0.25      0.44      0.32      3821

    accuracy                           0.76     29597
   macro avg       0.58      0.62      0.59     29597
weighted avg       0.82      0.76      0.79     29597


Threshold = 0.3


              precision    recall  f1-score   support

           0       0.90      0.86      0.88     25776
           1       0.28      0.37      0.32      3821

    accuracy                           0.80     29597
   macro avg       0.59      0.62      0.60     29597
weighted avg       0.82      0.80      0.81     29597


Threshold = 0.35


              precision    recall  f1-score   support

           0       0.90      0.90      0.90     25776
           1       0.31      0.31      0.31      3821

    accuracy                           0.82     29597
   macro avg       0.60      0.61      0.60     29597
weighted avg       0.82      0.82      0.82     29597



## 5) Model comparison


In [12]:
results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)
results_df


,model,roc_auc,pr_auc,brier
0,CatBoost,0.673175,0.296594,0.115504
1,XGBoost,0.679085,0.290261,0.179715
2,LogisticRegression,0.671615,0.278505,0.109457


## 6) Threshold selection for product decisions

Default threshold 0.5 is rarely optimal for imbalanced problems.
This section produces a small table of precision/recall trade-offs.

We need to choose a threshold based on product requirements, for example:
- Target recall (catch more delays) if missed delays are costly for users
- Target precision (reduce false alerts) if alerts are disruptive

In [13]:
def threshold_table(y_true, p, thresholds):
    rows = []
    for thr in thresholds:
        y_pred = (p >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        rows.append({
            "threshold": float(thr),
            "precision": precision,
            "recall": recall,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)
        })
    return pd.DataFrame(rows)

best_model = results_df.loc[0, "model"]
p_best = preds[best_model]

thresholds = np.round(np.linspace(0.10, 0.90, 9), 2)
thr_df = threshold_table(y_test, p_best, thresholds)
thr_df


,threshold,precision,recall,tp,fp,fn,tn
0,0.1,0.170095,0.748757,2861,13959,960,11817
1,0.2,0.222837,0.516880,1975,6888,1846,18888
2,0.3,0.278852,0.381314,1457,3768,2364,22008
3,0.4,0.324524,0.276367,1056,2198,2765,23578
4,0.5,0.380058,0.206490,789,1287,3032,24489
5,0.6,0.440031,0.147867,565,719,3256,25057
6,0.7,0.536137,0.106778,408,353,3413,25423
7,0.8,0.652913,0.070400,269,143,3552,25633
8,0.9,0.805556,0.045538,174,42,3647,25734


## 7) Probability calibration

If app displays probabilities (e.g., "Delay risk: 35%"), calibrate the chosen model.
Calibration improves probability reliability (often lowers Brier score).

- For LogisticRegression/XGBoost (sklearn pipeline): `CalibratedClassifierCV`
- For CatBoost: isotonic regression over predicted probabilities (simple, effective)



In [14]:
# Probability calibration (note)
# ------------------------------
# CatBoost probabilities are already calibrated via Platt scaling in the CatBoost cell above.
# If you want to calibrate LogisticRegression / XGBoost as well, use CalibratedClassifierCV with cv=5.
# (This is optional and can be skipped for now.)

p_cal = None
calibrated = None

if best_model in ["LogisticRegression", "XGBoost"]:
    base_model = lr if best_model == "LogisticRegression" else xgb
    calibrated = CalibratedClassifierCV(base_model, method="isotonic", cv=5)
    calibrated.fit(X_train, y_train)
    p_cal = calibrated.predict_proba(X_test)[:, 1]

    print("Uncalibrated:", compute_metrics(y_test, p_best))
    print("Calibrated:  ", compute_metrics(y_test, p_cal))
else:
    print("Calibration step skipped (CatBoost already calibrated with Platt).")


Calibration step skipped (CatBoost already calibrated with Platt).


## 8) Export model artifacts

This section exports:
- Model artifact (joblib for sklearn pipelines, `.cbm` for CatBoost)
- `model_metadata.json` containing:
  - selected feature list
  - chosen operating threshold
  - test metrics
  - delay rates

Set `OPERATING_THRESHOLD` based on the threshold table above.


In [15]:
if not ART_DIR:
    raise ValueError("ART_DIR is not set. Check your .env and load_dotenv().")

os.makedirs(ART_DIR, exist_ok=True)

OPERATING_THRESHOLD = 0.30  # choose from threshold table based on your product requirements

metadata = {
    "best_model": best_model,               # e.g. "CatBoost"
    "selected_features": selected,
    "categorical_features_lr_xgb": cat_cols,
    "operating_threshold": OPERATING_THRESHOLD,
    "metrics_test": results_df.loc[0].to_dict(),
    "delay_rate_train": float(y_train.mean()),
    "delay_rate_test": float(y_test.mean()),
}

# If CatBoost was trained with Platt calibration, record the artifact name
if best_model == "CatBoost" and platt is not None:
    metadata["calibration"] = {
        "method": "platt_sigmoid",
        "calibrator_path": "platt_calibrator.joblib"
    }

# Save metadata
if not META_PATH:
    META_PATH = os.path.join(ART_DIR, "model_metadata.json")

with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# Save model artifact
if best_model == "LogisticRegression":
    if not JOBLIB_PATH:
        JOBLIB_PATH = os.path.join(ART_DIR, "best_model.joblib")
    joblib.dump(lr, JOBLIB_PATH)
    print("Saved sklearn model:", JOBLIB_PATH)

elif best_model == "XGBoost" and xgb is not None:
    if not JOBLIB_PATH:
        JOBLIB_PATH = os.path.join(ART_DIR, "best_model.joblib")
    joblib.dump(xgb, JOBLIB_PATH)
    print("Saved XGBoost model:", JOBLIB_PATH)

elif best_model == "CatBoost" and cb is not None:
    # Save CatBoost
    if not CB_PATH:
        CB_PATH = os.path.join(ART_DIR, "catboost_model.cbm")
    cb.save_model(CB_PATH)
    print("Saved CatBoost model:", CB_PATH)

    # Save Platt calibrator alongside it (used by backend to calibrate probabilities)
    if platt is not None:
        CALIB_PATH = os.path.join(ART_DIR, "platt_calibrator.joblib")
        joblib.dump(platt, CALIB_PATH)
        print("Saved Platt calibrator:", CALIB_PATH)

print("Saved metadata:", META_PATH)


Saved CatBoost model: /Users/nikitha/Documents/flight-delay-prediction-ice/notebooks/model_artifacts_final/catboost_model.cbm
Saved Platt calibrator: /Users/nikitha/Documents/flight-delay-prediction-ice/notebooks/model_artifacts_final/platt_calibrator.joblib
Saved metadata: /Users/nikitha/Documents/flight-delay-prediction-ice/notebooks/model_artifacts_final/model_metadata.json


In [16]:
feature_store = {
    "global_delay_rate": float(y_train.mean()),
    "risk_maps": {
        "ORIGIN_RISK": train_df.set_index("ORIGIN")["ORIGIN_RISK"].to_dict(),
        "DEST_RISK": train_df.set_index("DEST")["DEST_RISK"].to_dict(),
        "ROUTE_RISK": train_df.set_index("ROUTE")["ROUTE_RISK"].to_dict(),
        "UNIQUE_CARRIER_RISK": train_df.set_index("UNIQUE_CARRIER")["UNIQUE_CARRIER_RISK"].to_dict(),
        "CARRIER_ROUTE_RISK": train_df.set_index("CARRIER_ROUTE")["CARRIER_ROUTE_RISK"].to_dict(),
    },
    "count_maps": {
        "TRAIN_ORIGIN_COUNT": train_df.set_index("ORIGIN")["TRAIN_ORIGIN_COUNT"].to_dict(),
        "TRAIN_DEST_COUNT": train_df.set_index("DEST")["TRAIN_DEST_COUNT"].to_dict(),
        "TRAIN_ROUTE_COUNT": train_df.set_index("ROUTE")["TRAIN_ROUTE_COUNT"].to_dict(),
        "TRAIN_UNIQUE_CARRIER_COUNT": train_df.set_index("UNIQUE_CARRIER")["TRAIN_UNIQUE_CARRIER_COUNT"].to_dict(),
        "TRAIN_CARRIER_ROUTE_COUNT": train_df.set_index("CARRIER_ROUTE")["TRAIN_CARRIER_ROUTE_COUNT"].to_dict(),
    }
}

with open(FEATURE_STORE_PATH, "w") as f:
    json.dump(feature_store, f, indent=2)

In [17]:
import json

# Create a dictionary to store our feature lookups
artifacts = {
    "means": {},
    "mappings": {},
    "defaults": {}
}

# Save Risk Encodings and Counts (using the train_df from your notebook)
# Map the raw value (e.g., 'ATL') to its Risk Score and Count
for col in ["ORIGIN", "DEST", "UNIQUE_CARRIER", "ROUTE", "CARRIER_ROUTE"]:
    # Risk Mapping
    if f"{col}_RISK" in train_df.columns:
        artifacts["mappings"][f"{col}_RISK"] = train_df.set_index(col)[f"{col}_RISK"].to_dict()
        artifacts["defaults"][f"{col}_RISK"] = train_df[f"{col}_RISK"].mean() # Fallback for unknown airports
        
    # Count Mapping
    if f"TRAIN_{col}_COUNT" in train_df.columns:
        artifacts["mappings"][f"TRAIN_{col}_COUNT"] = train_df.set_index(col)[f"TRAIN_{col}_COUNT"].to_dict()
        artifacts["defaults"][f"TRAIN_{col}_COUNT"] = 0

# Save Standardization Stats (Mean/Std) for Distance
artifacts["means"]["DISTANCE"] = train_df["DISTANCE"].mean()
artifacts["means"]["DISTANCE_STD"] = train_df["DISTANCE"].std()

# Save the artifacts to a JSON file
with open(LOOKUP_PATH, "w") as f:
    json.dump(artifacts, f)

print("Feature lookup tables saved successfully!")

Feature lookup tables saved successfully!
